In [0]:
SELECT
    ISBN,
    COUNT(*) AS rating_count,
    AVG(`Book-Rating`) AS avg_rating
FROM ratings
WHERE `Book-Rating` > 0
GROUP BY ISBN
ORDER BY avg_rating DESC;

ISBN,rating_count,avg_rating
3453092481,1,10.0
0312115695,1,10.0
0679811796,1,10.0
7119029770,1,10.0
0573030219,1,10.0
074970053X,1,10.0
0671824295,1,10.0
7108013134/J.,1,10.0
0689846258,1,10.0
711900557X,1,10.0


In [0]:
CREATE OR REPLACE VIEW book_rating_stats AS

SELECT
    ISBN,
    COUNT(*) AS rating_count,
    AVG(`Book-Rating`) AS avg_rating
FROM ratings
WHERE `Book-Rating` > 0
GROUP BY ISBN;

In [0]:
SELECT
    `Book-Rating`,
    COUNT(*) AS occurrences
FROM ratings
GROUP BY `Book-Rating`
ORDER BY `Book-Rating`;

Book-Rating,occurrences
0,716109
1,1770
2,2759
3,5996
4,8904
5,50974
6,36924
7,76457
8,103736
9,67541


In [0]:
SELECT
    MIN(rating_count) AS min_ratings,
    percentile_approx(rating_count, 0.25) AS p25,
    percentile_approx(rating_count, 0.50) AS median,
    percentile_approx(rating_count, 0.75) AS p75,
    percentile_approx(rating_count, 0.90) AS p90,
    percentile_approx(rating_count, 0.95) AS p95,
    percentile_approx(rating_count, 0.99) AS p99,
    MAX(rating_count) AS max_ratings
FROM book_rating_stats;

min_ratings,p25,median,p75,p90,p95,p99,max_ratings
1,1,1,2,4,6,22,707


In [0]:
-- Histogram: number of ratings per book (binned) vs number of books
SELECT
    CASE
        WHEN rating_count BETWEEN  1 AND  5 THEN '01-05'
        WHEN rating_count BETWEEN  6 AND 10 THEN '06-10'
        WHEN rating_count BETWEEN 11 AND 20 THEN '11-20'
        WHEN rating_count BETWEEN 21 AND 30 THEN '21-30'
        WHEN rating_count BETWEEN 31 AND 50 THEN '31-50'
        WHEN rating_count BETWEEN 51 AND 100 THEN '51-100'
        WHEN rating_count > 100 THEN '100+'
    END AS rating_count_bin,
    CASE
        WHEN rating_count BETWEEN  1 AND  5 THEN 1
        WHEN rating_count BETWEEN  6 AND 10 THEN 2
        WHEN rating_count BETWEEN 11 AND 20 THEN 3
        WHEN rating_count BETWEEN 21 AND 30 THEN 4
        WHEN rating_count BETWEEN 31 AND 50 THEN 5
        WHEN rating_count BETWEEN 51 AND 100 THEN 6
        WHEN rating_count > 100 THEN 7
    END AS bin_order,
    COUNT(*) AS number_of_books
FROM book_rating_stats
GROUP BY
    CASE
        WHEN rating_count BETWEEN  1 AND  5 THEN '01-05'
        WHEN rating_count BETWEEN  6 AND 10 THEN '06-10'
        WHEN rating_count BETWEEN 11 AND 20 THEN '11-20'
        WHEN rating_count BETWEEN 21 AND 30 THEN '21-30'
        WHEN rating_count BETWEEN 31 AND 50 THEN '31-50'
        WHEN rating_count BETWEEN 51 AND 100 THEN '51-100'
        WHEN rating_count > 100 THEN '100+'
    END,
    CASE
        WHEN rating_count BETWEEN  1 AND  5 THEN 1
        WHEN rating_count BETWEEN  6 AND 10 THEN 2
        WHEN rating_count BETWEEN 11 AND 20 THEN 3
        WHEN rating_count BETWEEN 21 AND 30 THEN 4
        WHEN rating_count BETWEEN 31 AND 50 THEN 5
        WHEN rating_count BETWEEN 51 AND 100 THEN 6
        WHEN rating_count > 100 THEN 7
    END
ORDER BY bin_order;

rating_count_bin,bin_order,number_of_books
01-05,1,174713
06-10,2,6297
11-20,3,2929
21-30,4,950
31-50,5,573
51-100,6,374
100+,7,137


In [0]:
SELECT
    rating_count            AS x_axis_number_of_ratings_per_book,
    COUNT(*)                AS y_axis_number_of_books
FROM book_rating_stats
GROUP BY rating_count
ORDER BY rating_count;

x_axis_number_of_ratings_per_book,y_axis_number_of_books
1,129621
2,26261
3,10173
4,5383
5,3275
6,2134
7,1497
8,1147
9,837
10,682


In [0]:
SELECT
    percentile_approx(rating_count, 0.25) AS p25,
    percentile_approx(rating_count, 0.50) AS median,
    percentile_approx(rating_count, 0.75) AS p75,
    percentile_approx(rating_count, 0.90) AS p90,
    percentile_approx(rating_count, 0.95) AS p95,
    percentile_approx(rating_count, 0.99) AS p99,
    MAX(rating_count) AS max_ratings
FROM book_rating_stats;

p25,median,p75,p90,p95,p99,max_ratings
1,1,2,4,6,22,707


In [0]:
SELECT
    COUNT(*) AS total_books,
    SUM(CASE WHEN rating_count >= 2 THEN 1 ELSE 0 END) AS at_least_2,
    SUM(CASE WHEN rating_count >= 3 THEN 1 ELSE 0 END) AS at_least_3,
    SUM(CASE WHEN rating_count >= 5 THEN 1 ELSE 0 END) AS at_least_5,
    SUM(CASE WHEN rating_count >= 10 THEN 1 ELSE 0 END) AS at_least_10,
    SUM(CASE WHEN rating_count >= 20 THEN 1 ELSE 0 END) AS at_least_20,
    SUM(CASE WHEN rating_count >= 50 THEN 1 ELSE 0 END) AS at_least_50,
    SUM(CASE WHEN rating_count >= 100 THEN 1 ELSE 0 END) AS at_least_100
FROM book_rating_stats;

total_books,at_least_2,at_least_3,at_least_5,at_least_10,at_least_20,at_least_50,at_least_100
185973,56352,30091,14535,5645,2179,537,140


In [0]:
SELECT
    CASE
        WHEN rating_count = 1              THEN '1'
        WHEN rating_count = 2              THEN '2'
        WHEN rating_count BETWEEN 3 AND 4   THEN '3-4'
        WHEN rating_count BETWEEN 5 AND 9   THEN '5-9'
        WHEN rating_count BETWEEN 10 AND 19  THEN '10-19'
        WHEN rating_count BETWEEN 20 AND 49  THEN '20-49'
        WHEN rating_count BETWEEN 50 AND 99  THEN '50-99'
        ELSE '100+'
    END AS rating_range,

    CASE
        WHEN rating_count = 1              THEN 1
        WHEN rating_count = 2              THEN 2
        WHEN rating_count BETWEEN 3 AND 4   THEN 3
        WHEN rating_count BETWEEN 5 AND 9   THEN 4
        WHEN rating_count BETWEEN 10 AND 19  THEN 5
        WHEN rating_count BETWEEN 20 AND 49  THEN 6
        WHEN rating_count BETWEEN 50 AND 99  THEN 7
        ELSE 8
    END AS bin_order,

    COUNT(*) AS number_of_books

FROM book_rating_stats

GROUP BY
    CASE
        WHEN rating_count = 1              THEN '1'
        WHEN rating_count = 2              THEN '2'
        WHEN rating_count BETWEEN 3 AND 4   THEN '3-4'
        WHEN rating_count BETWEEN 5 AND 9   THEN '5-9'
        WHEN rating_count BETWEEN 10 AND 19  THEN '10-19'
        WHEN rating_count BETWEEN 20 AND 49  THEN '20-49'
        WHEN rating_count BETWEEN 50 AND 99  THEN '50-99'
        ELSE '100+'
    END,
    CASE
        WHEN rating_count = 1              THEN 1
        WHEN rating_count = 2              THEN 2
        WHEN rating_count BETWEEN 3 AND 4   THEN 3
        WHEN rating_count BETWEEN 5 AND 9   THEN 4
        WHEN rating_count BETWEEN 10 AND 19  THEN 5
        WHEN rating_count BETWEEN 20 AND 49  THEN 6
        WHEN rating_count BETWEEN 50 AND 99  THEN 7
        ELSE 8
    END

ORDER BY bin_order;

rating_range,bin_order,number_of_books
1,1,129621
2,2,26261
3-4,3,15556
5-9,4,8890
10-19,5,3466
20-49,6,1642
50-99,7,397
100+,8,140


In [0]:
CREATE OR REPLACE TEMP VIEW eligible_books AS
SELECT
    ISBN,
    rating_count,
    avg_rating
FROM book_rating_stats
WHERE rating_count >= 5;

In [0]:
SELECT
    MIN(`Year-Of-Publication`) AS min_year,
    MAX(`Year-Of-Publication`) AS max_year
FROM books;

min_year,max_year
1961,"and the Final Ride of Jimmy Hoffa"""""""


In [0]:
DESCRIBE books;

col_name,data_type,comment
ISBN,string,null
Book-Title,string,null
Book-Author,string,null
Year-Of-Publication,string,null
Publisher,string,null
Image-URL-S,string,null
Image-URL-M,string,null
Image-URL-L,string,null
_rescued_data,string,null


In [0]:
SELECT
    ISBN,
    `Book-Title`,
    `Book-Author`,
    `Year-Of-Publication`,
    Publisher
FROM books
WHERE TRY_CAST(`Year-Of-Publication` AS INT) IS NULL;

ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0871564599,"""The War Against the Greens: The \Wise-Use"""" Movement",the New Right,"and Anti-Environmental Violence""""""",David Helvarg
0373168705,"""The Virgin Bride Said, \Wow!"""" (The Lockharts Of Texas) (Harlequin American Romance","No 870)""""""",Cathy Thacker,2001
0451527070,"""Walden Or, Life in the Woods and \on the Duty of Civil Disobedience"""": And","on the Duty of Civil Disobedience (Signet Classics (Paperback))""""""",Henry David Thoreau,1999
0373258267,"""Say \Ahhh ..."""" (Bedside Manners) (Harlequin Temptation","726)""""""",Donna Sterling,1999
1853670383,"""King of Air Fighters: The Biography of Major \Mick"""" Mannock",V.C.,D.S.O.,M.C. (Vintage Aviation Library
0192834843,"\A Room of One's Own""","and \""""Three Guineas\"""" (Oxford World's Classics)""""""",Virginia Woolf,1998
326800105X,"Wahlfamilie: Die \Wohngemeinschaft Jung und Alt""","ein Modell fÃ¼r das Wohnen von morgen""""""",Ulrich Schmidt,1990
1586420771,"""I Heard You Paint Houses : Frank \The Irishman"""" Sheeran and the Inside Story of the Mafia",the Teamsters,"and the Final Ride of Jimmy Hoffa""""""",CHARLES BRANDT
157243144X,"""\When I Was Your Age..."""": Remarkable Achievements of Writers",Artists,"and Musicians at Every Age from 1 to 100""""""",David Lewman
1885222971,"""The Lion King: The Pal Patrol (Disney's \Storytime Treasures"""" Library","Volume 1)""""""",Inc. Staff Disney Enterprises,1997


In [0]:
SELECT
    ISBN,
    `Book-Title`,
    `Book-Author`,
    `Year-Of-Publication`,
    Publisher
FROM books
WHERE TRY_CAST(`Year-Of-Publication` AS INT) IS NULL
LIMIT 20;

ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0871564599,"""The War Against the Greens: The \Wise-Use"""" Movement",the New Right,"and Anti-Environmental Violence""""""",David Helvarg
0373168705,"""The Virgin Bride Said, \Wow!"""" (The Lockharts Of Texas) (Harlequin American Romance","No 870)""""""",Cathy Thacker,2001
0451527070,"""Walden Or, Life in the Woods and \on the Duty of Civil Disobedience"""": And","on the Duty of Civil Disobedience (Signet Classics (Paperback))""""""",Henry David Thoreau,1999
0373258267,"""Say \Ahhh ..."""" (Bedside Manners) (Harlequin Temptation","726)""""""",Donna Sterling,1999
1853670383,"""King of Air Fighters: The Biography of Major \Mick"""" Mannock",V.C.,D.S.O.,M.C. (Vintage Aviation Library
0192834843,"\A Room of One's Own""","and \""""Three Guineas\"""" (Oxford World's Classics)""""""",Virginia Woolf,1998
326800105X,"Wahlfamilie: Die \Wohngemeinschaft Jung und Alt""","ein Modell fÃ¼r das Wohnen von morgen""""""",Ulrich Schmidt,1990
1586420771,"""I Heard You Paint Houses : Frank \The Irishman"""" Sheeran and the Inside Story of the Mafia",the Teamsters,"and the Final Ride of Jimmy Hoffa""""""",CHARLES BRANDT
157243144X,"""\When I Was Your Age..."""": Remarkable Achievements of Writers",Artists,"and Musicians at Every Age from 1 to 100""""""",David Lewman
1885222971,"""The Lion King: The Pal Patrol (Disney's \Storytime Treasures"""" Library","Volume 1)""""""",Inc. Staff Disney Enterprises,1997


In [0]:
CREATE OR REPLACE TEMP VIEW clean_books AS
SELECT
    ISBN,
    `Book-Title`,
    `Book-Author`,
    CAST(`Year-Of-Publication` AS INT) AS publication_year,
    Publisher
FROM books
WHERE TRY_CAST(`Year-Of-Publication` AS INT) IS NOT NULL;

In [0]:
SELECT
    MIN(publication_year) AS min_year,
    MAX(publication_year) AS max_year
FROM clean_books;

min_year,max_year
0,2050


In [0]:
SELECT
    COUNT(*) AS invalid_year_books
FROM clean_books
WHERE publication_year < 1800
   OR publication_year > 2026;

invalid_year_books
4631


In [0]:
SELECT
    publication_year,
    COUNT(*) AS number_of_books
FROM clean_books
WHERE publication_year < 1800
GROUP BY publication_year
ORDER BY publication_year;

publication_year,number_of_books
0,4618
1376,1
1378,1


In [0]:
SELECT
    ISBN,
    `Book-Title`,
    `Book-Author`,
    publication_year,
    Publisher
FROM clean_books
WHERE publication_year < 1800
ORDER BY publication_year;

ISBN,Book-Title,Book-Author,publication_year,Publisher
0880299517,Georgia Okeeffe,Elizabet Montgomery,0,Barnes Noble
073570970X,Php Functions Essential Reference (Essential Reference),Zak Greant,0,Sams
0722522606,Aromatherapy for Women How to Use Essent,Maggie Tisserand,0,Thorsons
0002169290,For Goodness Sake,Merren Parker,0,Harpercollins Publisher
2253043133,Le Shanghaie,Le Gal,0,Livre de Poche
0006380263,Perfection of the Morning an Apprentices,Sharon Butala,0,Harperperennial
0376029625,Wok Cookbook,Sunset Ed,0,Sunset Publishing
0718143396,At All Costs,John Gilstrap,0,Penguin Putnam~trade
0330324632,On the Road to Baghdad,Guneli Gun,0,Pan Books Ltd
3548301231,Jane Eyre,C Bronte,0,"Ullstein-Taschenbuch-Verlag, Zweigniederlassung der Ullstein Buchverlage GmbH"


In [0]:
CREATE OR REPLACE TEMP VIEW clean_books AS
SELECT
    ISBN,
    `Book-Title`,
    `Book-Author`,
    CASE
        WHEN TRY_CAST(`Year-Of-Publication` AS INT) BETWEEN 1800 AND 2026
        THEN TRY_CAST(`Year-Of-Publication` AS INT)
        ELSE NULL
    END AS publication_year,
    Publisher
FROM books
WHERE TRY_CAST(`Year-Of-Publication` AS INT) IS NOT NULL;

In [0]:
CREATE OR REPLACE VIEW eligible_books AS
SELECT
    ISBN,
    rating_count,
    avg_rating
FROM book_rating_stats
WHERE rating_count >= 5;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7177267687050248>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE VIEW eligible_books AS\nSELECT\n    ISBN,\n    rating_count,\n    avg_rating\nFROM book_rating_stats\nWHERE rating_count >= 5;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in

In [0]:
SELECT COUNT(*) AS eligible_books
FROM eligible_books;

eligible_books
14535


In [0]:
SELECT AVG(avg_rating) AS global_mean
FROM eligible_books;

global_mean
7.670442132083944


In [0]:
CREATE OR REPLACE TEMP VIEW book_quality_scores AS
SELECT
    ISBN,
    rating_count,
    avg_rating,
    (
        (rating_count / (rating_count + 5.0)) * avg_rating
        +
        (5.0 / (rating_count + 5.0)) * 7.6704
    ) AS weighted_rating
FROM eligible_books;

In [0]:
SELECT
    b.`Book-Title`,
    b.`Book-Author`,
    q.ISBN,
    q.rating_count,
    ROUND(q.avg_rating, 2) AS avg_rating,
    ROUND(q.weighted_rating, 2) AS weighted_rating
FROM book_quality_scores q
JOIN books b
    ON q.ISBN = b.ISBN
ORDER BY q.weighted_rating DESC
LIMIT 30;

Book-Title,Book-Author,ISBN,rating_count,avg_rating,weighted_rating
Harry Potter and the Chamber of Secrets Postcard Book,J. K. Rowling,0439425220,23,9.87,9.48
"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,0618002235,25,9.72,9.38
The Giving Tree,Shel Silverstein,0060256656,20,9.75,9.33
Dilbert: A Book of Postcards,Scott Adams,0836213319,13,9.92,9.3
"The Return of the King (The Lord of the Rings, Part 3)",J.R.R. TOLKIEN,0345339738,77,9.4,9.3
Postmarked Yesteryear: 30 Rare Holiday Postcards,Pamela E. Apkarian-Russell,1888054557,11,10.0,9.27
Calvin and Hobbes,Bill Watterson,0836220889,24,9.58,9.25
Fox in Socks (I Can Read It All by Myself Beginner Books),Dr. Seuss,0394800389,14,9.79,9.23
The Authoritative Calvin and Hobbes (Calvin and Hobbes),Bill Watterson,0836218221,20,9.6,9.21
Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0439139597,137,9.26,9.21


am constatat ca exista 2 cazuri de carti cu dubluri, insa, fiind isbn uri diferite, nu se considera o eroare deoarece ar putea fi 2 editii diferite ale aceleiasi carti - solutia: nu o sa stergem unul dintre entry-urile cartilor cu dubluri, dar vom agrega recenziile fiindca asta este informatia utila pentru client. - acest lucru se va rezolva cand voi include datsets externe
am gasit si erori de scriere cum ar fi 'Antoine de Saint-ExupÃ©ry' si 'Ã?Â?berlebenden'

In [0]:
SELECT
    b.`Book-Title`,
    b.`Book-Author`,
    q.rating_count,
    ROUND(q.avg_rating, 2) AS avg_rating,
    ROUND(q.weighted_rating, 2) AS weighted_rating
FROM book_quality_scores q
JOIN books b
    ON q.ISBN = b.ISBN
ORDER BY q.weighted_rating DESC
LIMIT 10;

Book-Title,Book-Author,rating_count,avg_rating,weighted_rating
Harry Potter and the Chamber of Secrets Postcard Book,J. K. Rowling,23,9.87,9.48
"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,25,9.72,9.38
The Giving Tree,Shel Silverstein,20,9.75,9.33
Dilbert: A Book of Postcards,Scott Adams,13,9.92,9.3
"The Return of the King (The Lord of the Rings, Part 3)",J.R.R. TOLKIEN,77,9.4,9.3
Postmarked Yesteryear: 30 Rare Holiday Postcards,Pamela E. Apkarian-Russell,11,10.0,9.27
Calvin and Hobbes,Bill Watterson,24,9.58,9.25
Fox in Socks (I Can Read It All by Myself Beginner Books),Dr. Seuss,14,9.79,9.23
The Authoritative Calvin and Hobbes (Calvin and Hobbes),Bill Watterson,20,9.6,9.21
Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,137,9.26,9.21


In [0]:
CREATE OR REPLACE TEMP VIEW book_rating_stats AS
SELECT
    ISBN,
    COUNT(*) AS rating_count,
    AVG(`Book-Rating`) AS avg_rating
FROM ratings
WHERE `Book-Rating` > 0
GROUP BY ISBN;

In [0]:
SELECT
    CASE
        WHEN rating_count = 1 THEN '1'
        WHEN rating_count BETWEEN 2 AND 5 THEN '2-5'
        WHEN rating_count BETWEEN 6 AND 10 THEN '6-10'
        WHEN rating_count BETWEEN 11 AND 25 THEN '11-25'
        WHEN rating_count BETWEEN 26 AND 50 THEN '26-50'
        WHEN rating_count BETWEEN 51 AND 100 THEN '51-100'
        ELSE '100+'
    END AS rating_count_range,
    COUNT(*) AS number_of_books
FROM book_rating_stats
GROUP BY rating_count_range;

rating_count_range,number_of_books
1,129621
2-5,45092
51-100,374
26-50,936
11-25,3516
6-10,6297
100+,137


In [0]:
SHOW TABLES IN workspace.default LIKE 'book_quality_scores'

database,tableName,isTemporary
,book_quality_scores,true


In [0]:
SELECT *
FROM book_quality_scores
ORDER BY weighted_rating DESC
LIMIT 5;

ISBN,rating_count,avg_rating,weighted_rating
0439425220,23,9.869565217391305,9.476857142857144
0618002235,25,9.72,9.378400000000001
0060256656,20,9.75,9.33408
8826703132,21,9.714285714285714,9.32123076923077
0836213319,13,9.923076923076923,9.297333333333334


In [0]:
SHOW TABLES IN workspace.default LIKE 'book_quality_scores'

database,tableName,isTemporary
,book_quality_scores,true


In [0]:
CREATE OR REPLACE VIEW workspace.default.book_quality_scores AS
SELECT
    ISBN,
    rating_count,
    avg_rating,
    (
        (rating_count / (rating_count + 5.0)) * avg_rating
        +
        (5.0 / (rating_count + 5.0)) * 7.6704
    ) AS weighted_rating
FROM workspace.default.eligible_books;

In [0]:
CREATE OR REPLACE VIEW workspace.default.book_rating_stats AS
SELECT
    ISBN,
    COUNT(*) AS rating_count,
    AVG(`Book-Rating`) AS avg_rating
FROM workspace.default.ratings
WHERE `Book-Rating` > 0
GROUP BY ISBN;

In [0]:
CREATE OR REPLACE VIEW workspace.default.eligible_books AS
SELECT
    ISBN,
    rating_count,
    avg_rating
FROM workspace.default.book_rating_stats
WHERE rating_count >= 5;